# Lesson 76 · Music theory crash course — 88 piano keys, only 12 notes: pitch, intervals, the chroma wheel and equal temperament (12-TET)

**Learning goals**
- Understand the 12 pitch classes and octave equivalence
- Master the MIDI ↔ frequency conversion formula
- Understand intervals: the distance in semitones between two notes, and the common interval names
- Build a chroma wheel visualization
- See why audio AI uses chroma instead of raw frequency

> **Opening lesson**: put "12 pitch classes + octave equivalence" and $f = 440\cdot 2^{n/12}$ first; for harmonics, look back at L07.

← **Previous lesson** · [L75 · The ASR pipeline drawn](../7_asr/L75_visual_asr.ipynb)

> Last lesson covered **the ASR pipeline drawn**: visualizing the road waveform → spectrogram → tokens → text.
> This lesson explores **a music theory crash course**.

## This lesson's story: why do C4 and C5 sound like "the same note" when they aren't the same frequency?

A piano has 88 keys, but music has only **12 pitch classes**, repeating over and over. Middle C (C4) = 261 Hz, the C an octave up (C5) = 523 Hz — the frequency doubles, yet the ear hears "the same note" (just higher).

**This is no coincidence; it's physics**: when a string's vibration frequency doubles, the overtone series lines up completely, and the brain files the two notes in the same category. This is called "octave equivalence".

**The MIDI numbering system**: A4 (concert A) = MIDI 69 = 440 Hz. Each semitone is one MIDI number apart; each octave is 12 apart.

**Frequency formula**:
```
f = 440 × 2^((midi - 69) / 12)
```
- `midi=69` → f = 440 Hz (A4)
- `midi=60` → f = 261.6 Hz (C4, middle C)
- `midi=81` → f = 880 Hz (A5, one octave up)

**Pitch class (chroma)**: "fold" the MIDI number down to 0–11:
```
chroma = midi % 12   (MIDI 60 = C4 and 60 % 12 = 0, so C=0, C#=1, ..., B=11 comes for free — no offset needed)
```

The chroma wheel is the 12 pitch classes arranged in a circle — neighbors are a semitone apart, and opposite points are a tritone apart. All chord, key and harmony analysis is built on this ring.

Your task in this section: implement two conversion functions, and use maths to confirm your musical intuition.

## Intervals: the distance between two notes

An **interval** = the number of **semitones** between two notes; in MIDI space it is simply the absolute difference of the two numbers.

| Interval name | Semitones | Example (starting from C) |
|---|:---:|---|
| minor 2nd | 1 | C → C# |
| major 2nd | 2 | C → D |
| minor 3rd | 3 | C → D# |
| major 3rd | 4 | C → E |
| perfect 4th | 5 | C → F |
| augmented 4th / tritone | 6 | C → F# |
| perfect 5th | 7 | C → G |
| major 6th | 9 | C → A |
| octave | 12 | C → C (an octave up) |

**MIDI interval formula**: `interval_semitones = |MIDI_high - MIDI_low|`

Example: C4 (MIDI 60) to G4 (MIDI 67) = 7 semitones = a perfect fifth.

> **Think about it**: how many semitones lie between A3 (MIDI 57) and E4 (MIDI 64)? What interval is that?
> (Answer: |64-57| = 7 semitones = a perfect fifth)


In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PITCH_CLASSES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
MIDI_A4, FREQ_A4 = 69, 440.0

## ✏️ Task 1: MIDI ↔ frequency conversion

**Signature**:
```python
def midi_to_freq(midi: float) -> float:
    # returns: frequency in Hz; A4=69→440Hz
```

**Formula**: `f = 440 × 2^((midi - 69) / 12)`

| MIDI | Note name | Expected frequency |
|---|---|---|
| 69 | A4 | 440.00 Hz |
| 60 | C4 (middle C) | 261.63 Hz |
| 81 | A5 | 880.00 Hz |
| 57 | A3 | 220.00 Hz |

**Acceptance criteria**:
- `abs(midi_to_freq(69) - 440.0) < 0.01`
- `abs(midi_to_freq(81) / midi_to_freq(69) - 2.0) < 1e-9` (one octave up = double the frequency)

In [ ]:
def midi_to_freq(midi: float) -> float:
    """MIDI number → frequency in Hz. A4=69=440Hz."""
    # ✏️ TODO: f(n) = 440 × 2^((n-69)/12)
    raise NotImplementedError("TODO")

def freq_to_midi(freq: float) -> float:
    """Frequency in Hz → floating-point MIDI number."""
    # ✏️ TODO: inverse formula: n = 69 + 12 × log2(f / 440)
    raise NotImplementedError("TODO")

# Check
for name, midi, expected in [("A4",69,440.0),("C4",60,261.63),("A3",57,220.0)]:
    got = midi_to_freq(midi)
    ok = abs(got - expected) < 0.5
    print(f"{name} MIDI{midi}: {got:.2f} Hz  {chr(9989) if ok else chr(10060)}")
    assert ok, f"{name} MIDI{midi}: expected {expected:.2f} Hz, got {got:.2f} Hz — check the formula f(n)=440×2^((n-69)/12)"

assert abs(freq_to_midi(440.0) - 69) < 0.01, "freq_to_midi(440) should return 69"
assert abs(midi_to_freq(0) - 8.18) < 0.1, "MIDI 0 (the lowest note) should be about 8.18 Hz"
print("✅ Frequency conversion checks passed")


## ✏️ Task 2: chroma_from_freq

**Signature**:
```python
def chroma_from_freq(freq: float) -> int:
    # returns: 0-11 (C=0, C#=1, D=2, ..., B=11)
```

**A 3-step route to the implementation**:

| Step | Operation | Formula |
|---|---|---|
| 1 | Frequency → MIDI number (the inverse function) | `midi = 69 + 12 × log2(freq / 440)` |
| 2 | MIDI → pitch class (fold away the octave) | `chroma_raw = round(midi) % 12` |
| 3 | No offset needed (C=0 holds automatically) | MIDI 60 (C4) % 12 = 0; self-check: A4=69 → 69%12=9=A |

**Acceptance criteria**:
- `chroma_from_freq(440.0)` == 9 (A4 → A = chroma 9)
- `chroma_from_freq(261.63)` == 0 (C4 → C = chroma 0)
- `chroma_from_freq(880.0)` == 9 (A5 has the same chroma as A4)

In [ ]:
def chroma_from_freq(freq: float) -> int:
    """Frequency in Hz → pitch class 0-11 (C=0, C#=1, ..., B=11)."""
    # ✏️ TODO: convert with freq_to_midi, round() to the nearest integer, then take mod 12
    # ⚠️ You must use round(), not int() (which truncates):
    #   B4=493.88Hz → freq_to_midi ≈ 70.9999
    #   int(70.9999)=70 → A# (wrong), round(70.9999)=71 → B (right) ✓
    # Reference implementation: n = round(freq_to_midi(freq)); return n % 12
    raise NotImplementedError("TODO")

# Check (includes the B4 rounding edge case)
for name, freq, expected_pc in [("C4",261.63,0),("A4",440.0,9),("A5",880.0,9),("B4",493.88,11)]:
    pc = chroma_from_freq(freq)
    print(f"{name} ({freq:.1f}Hz): chroma={pc} ({PITCH_CLASSES[pc]}) {chr(9989) if pc==expected_pc else chr(10060)}")
    assert pc == expected_pc, (
        f"{name} chroma wrong: expected {expected_pc} ({PITCH_CLASSES[expected_pc]}), "
        f"got {pc} — make sure you used round(), not int()"
    )

print("✅ chroma checks passed (including the B4 rounding edge case)")


In [ ]:
from pathlib import Path
# 🎨 Chroma wheel visualization (a teaching figure; aviz does not cover this)
fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)
colors = plt.cm.hsv(np.linspace(0, 1, 12, endpoint=False))
ax.bar(angles, np.ones(12), width=2*np.pi/12, color=colors, alpha=0.75, align='edge')
for i, (a, name) in enumerate(zip(angles + np.pi/12, PITCH_CLASSES)):
    ax.text(a, 1.35, name, ha='center', va='center', fontsize=10, fontweight='bold')
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.set_title('Chroma wheel — the 12 pitch classes', pad=15)
plt.tight_layout()
plt.savefig(Path('chroma_wheel.png'), dpi=80, bbox_inches='tight')
plt.show()

## The major scale and chroma

C major = C-D-E-F-G-A-B = chroma {0,2,4,5,7,9,11}.

Transpose a piece up or down into another key, and the whole chroma vector rotates.
Songs in different keys have chroma distributions that look different but follow a similar "pattern".

That is why chroma vectors allow **key-independent** music similarity comparison.

## Wrap-up

| Concept | Remember |
|---|---|
| 12-EDO | octave = 12 semitones; the frequency ratio between neighboring semitones = 2^(1/12) ≈ 1.0595 |
| MIDI | f(n)=440×2^((n-69)/12), A4 = MIDI 69 = 440 Hz |
| Chroma | MIDI mod 12: throw away the octave, keep the pitch class |
| L77 | apply chroma on top of the STFT to get a time-varying chromagram, using aurora.music |

The next lesson (L77) uses Aurora's music module to extract chroma, RMS, ZCR and other features, turning music theory into numeric vectors you can compute with.

## ✏️ Whiteboard challenge: pitch maths by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: What note is MIDI=72 (C4=60, 12 semitones per octave)? What is its frequency?
(72 = 60+12 = C5 (one octave up); f = 440×2^((72-69)/12) = 440×2^(3/12) ≈ 523.3 Hz)

**Question 2**: What is the name of the 440 Hz note? What is its chroma value?
(A4, chroma = 9; A = la = MIDI 69, 69 % 12 = 9; with C=0, A=9)

**Question 3**: The interval called a "perfect fifth" = 7 semitones. What is the frequency ratio C4→G4?
(2^(7/12) ≈ 1.498 ≈ 3/2 — this is the physical reason a perfect fifth sounds "consonant")

**Question 4**: If a piece is transposed from C major to D major (+2 semitones), how do all the MIDI numbers change?
(Every MIDI number goes up by 2; frequencies are multiplied by 2^(2/12) ≈ 1.122; the chroma shifts 2 places to the right)

**Question 5**: On the chroma wheel, how many semitones apart are C and F# (F sharp)? What is this interval called?
(C=0, F#=6, 6 semitones apart; this is the "tritone", the most dissonant interval, and it splits the octave exactly in half)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: frequency of MIDI=72
try:
    freq_72 = midi_to_freq(72)
    expected_72 = 440 * 2 ** ((72 - 69) / 12)
    assert abs(freq_72 - expected_72) < 0.01, f"midi_to_freq(72)={freq_72:.2f}, expected {expected_72:.2f}"
    print(f"Q1 ✅  midi_to_freq(72=C5)={freq_72:.2f} Hz (≈523.25 Hz)")
except (NotImplementedError, TypeError):
    expected_72 = 440 * 2 ** ((72 - 69) / 12)
    print(f"Q1 ✅  C5(MIDI=72): f=440×2^(3/12)≈{expected_72:.2f} Hz (will be verified once implemented)")

# Q2: chroma of 440 Hz
try:
    chroma_a4 = chroma_from_freq(440.0)
    assert chroma_a4 == 9, f"440Hz→chroma should be 9 (A), got {chroma_a4}"
    print(f"Q2 ✅  chroma_from_freq(440Hz=A4)={chroma_a4} (A=chroma 9, C=0)")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  440Hz=A4, chroma=9 (MIDI 69, 69%12=9; with C=0, A=9) (will be verified once implemented)")

# Q3: frequency ratio of a perfect fifth
perfect_fifth_ratio = 2 ** (7/12)
just_fifth = 3/2
print(f"Q3 ✅  perfect fifth (7 semitones) frequency ratio=2^(7/12)≈{perfect_fifth_ratio:.6f}, just intonation=3/2={just_fifth:.6f}, difference={abs(perfect_fifth_ratio-just_fifth)*100:.3f}%")
assert abs(perfect_fifth_ratio - just_fifth) < 0.01, "the 12-TET perfect fifth should be within 1% of just intonation"

# Q4: check transposition
try:
    # D major = C major + 2 semitones
    c_major = [60, 62, 64, 65, 67, 69, 71, 72]  # MIDI C major
    d_major_midi = [m + 2 for m in c_major]
    d_major_freq = [midi_to_freq(m) for m in d_major_midi]
    c_major_freq = [midi_to_freq(m) for m in c_major]
    ratios = [df/cf for df, cf in zip(d_major_freq, c_major_freq)]
    expected_ratio = 2 ** (2/12)
    assert all(abs(r - expected_ratio) < 1e-9 for r in ratios), "every frequency ratio should be 2^(2/12)"
    print(f"Q4 ✅  transpose +2 semitones: every frequency ×{expected_ratio:.4f} (=2^(2/12))")
except (NotImplementedError, TypeError):
    print(f"Q4 ✅  transpose +2 semitones: every MIDI +2, frequency ×2^(2/12)≈1.122 (will be verified once implemented)")

# Q5: the tritone
c_chroma, f_sharp_chroma = 0, 6
interval = abs(f_sharp_chroma - c_chroma)
assert interval == 6, f"C to F# should be 6 semitones, got {interval}"
print(f"Q5 ✅  C(chroma={c_chroma}) to F#(chroma={f_sharp_chroma})={interval} semitones = a tritone (splits the octave in half; the most dissonant interval)")
print("\n🎉 Pitch-maths whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l76_review = {
    "octave_equivalence":      None,  # understand octave equivalence: double the frequency → same note name, chroma=midi%12? True/False
    "midi_freq_formula":       None,  # remember f=440×2^((midi-69)/12) and can work out A3/C4/C5 by hand? True/False
    "midi_to_freq_impl":       None,  # midi_to_freq() implemented correctly, all 3 acceptance criteria pass? True/False
    "chroma_from_freq_impl":   None,  # chroma_from_freq() implemented correctly: A4→9, C4→0, A5→9? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard questions worked by hand? True/False
}

unfilled = [k for k, v in l76_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l76_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L76 complete! On to L77: music features')

---

→ **Next lesson** · [L77 · Music features](L77_music_features.ipynb)

> Next lesson covers **music features**: chroma, RMS energy and ZCR, implemented from scratch in aurora.music.